# Part 2 — Customer Churn Modeling

This notebook runs the reusable training pipeline implemented in `src/churn_model.py`. It compares Logistic Regression with LightGBM, selects by validation PR-AUC, tunes a threshold on the validation set, and reserves the test set for final evaluation.

Before running, install `requirements-modeling.txt`. If Part 1's feature CSV is missing, the training code attempts to run Part 1 first (internet is required for the first dataset download).

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown

from src.churn_model import read_feature_data, train_and_evaluate
from src.config import REPORTS_DIR, FIGURES_DIR, FEATURE_DATA_PATH

## 1. Check the modeling dataset

Review the row count, target balance, columns, and target definition before fitting anything.

In [ ]:
df = read_feature_data()
print(f'Shape: {df.shape}')
print(f'Churn rate: {df["Churn"].mean():.2%}')
display(df.head())
display(df['Churn'].value_counts().rename(index={0: 'Stayed', 1: 'Churned'}).to_frame('customers'))

## 2. Train and evaluate candidate models

The pipeline uses a stratified 60/20/20 train/validation/test split. It selects by validation PR-AUC and selects the classification threshold using validation F1. The held-out test set is not used for these decisions.

In [ ]:
result = train_and_evaluate(df)
display(result['comparison'])
display(pd.DataFrame(result['test_metrics']['test_metrics_at_selected_threshold'], index=['held-out test']).T)

## 3. Inspect detailed evaluation reports

The comparison and JSON metrics are saved under `reports/`. The tuned threshold is a baseline decision rule; choose a final business threshold after considering retention outreach costs and capacity.

In [ ]:
comparison = pd.read_csv(REPORTS_DIR / 'churn_model_comparison.csv')
threshold_scan = pd.read_csv(REPORTS_DIR / 'churn_threshold_analysis.csv')
metrics = json.loads((REPORTS_DIR / 'churn_test_metrics.json').read_text(encoding='utf-8'))
display(comparison)
display(threshold_scan.sort_values('f1', ascending=False).head(10))
display(pd.DataFrame(metrics['test_metrics_at_0_5'], index=['threshold_0_50']).T)
display(pd.DataFrame(metrics['test_metrics_at_selected_threshold'], index=['selected_threshold']).T)

## 4. Plot review

Confusion matrix helps inspect missed churners and false alarms. PR-AUC is useful under class imbalance; no single score is sufficient.

In [ ]:
for filename in ['part2_confusion_matrix.png', 'part2_precision_recall_curve.png', 'part2_roc_curve.png', 'part2_feature_importance.png']:
    image_path = FIGURES_DIR / filename
    if image_path.exists():
        display(Markdown(f'### {filename.replace("_", " ")}'))
        display(Image(filename=str(image_path)))

## 5. Run inference with the saved pipeline

A fresh customer CSV must include every feature column from `models/churn_model_metadata.json`. The script writes predictions to `reports/churn_predictions.csv`. This sample dataset is cross-sectional, so scores should be treated as a modeling demonstration and validated on time-separated company data before use in real decisions.

From the VS Code terminal, run:
```powershell
python -m src.predict_churn
```